# Assumption 1: Does transformer always make the same results?

In [3]:
import torch
import torch.nn as nn
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0
import torchvision.transforms as transforms
from tqdm.notebook import tqdm
import numpy as np
import os

# Add ../ as a directory to import from
import sys
sys.path.append('../')

from string_to_xml_to_vec import string2vec, vec2string, vec2xml, pretty_print_xml

In [2]:
from torch.utils.data import DataLoader
from plant_dataset import PlantDataset 
from plant_tokenizer import PAD_token

transform = transforms.Compose([
    #transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5, 0.5])
    #transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5,  0.5])
])



import pickle
# Check if the .plk file exists
dataset_dir = "../data/generated_dataset_Sep22_black"
saved_train_dataset_name = os.path.join(dataset_dir,"train_dataset.pkl")
saved_val_dataset_name = os.path.join(dataset_dir,"val_dataset.pkl")
if os.path.exists(saved_train_dataset_name):
    if os.path.exists(saved_train_dataset_name):
        print("Loading plant dataset from .pkl file")
        with open(saved_train_dataset_name, "rb") as f:
            train_dataset = pickle.load(f)
        with open(saved_val_dataset_name, "rb") as f:
            val_dataset = pickle.load(f)
else:
    preload = False

    train_dataset = PlantDataset(dataset_dir, plot=["000", "001", "002",], transform=transform, use_depth=True, preload=preload)
    #train_dataset = PlantDataset("../_data/Syn2Real_cowpea/Syn2Real_cowpea", plot=["000"], transform=transform)
    val_dataset = PlantDataset(dataset_dir, plot=["003"], transform=transform, use_depth=True, preload=preload)

    if preload:
        with open(saved_train_dataset_name, "wb") as f:
            pickle.dump(train_dataset, f)
        with open(saved_val_dataset_name, "wb") as f:
            pickle.dump(val_dataset, f)

Total 4831 images and plant strings loaded
Total 1585 images and plant strings loaded


In [ ]:
from models.model import ImageToSequenceTransformer

# Initialize model
num_layers =  3 # Default 6
num_heads = 4   # Default 8 
seq_dim = 43 # 43개의 토큰 
seq_embedding_dim = 64 # 
param_dim = 5 + 4 + 3 + 4 # 5 for shoot, 4 for the internode, 3 for the petiole, 4 for the leaf
param_embedding_dim = 64

model = ImageToSequenceTransformer(seq_embedding_dim=seq_embedding_dim, 
                                   param_embedding_dim=param_embedding_dim,
                                   num_layers=num_layers, num_heads=num_heads, 
                                   num_tokens=seq_dim, num_params=param_dim,
                                   decoder_only=True,
                                   use_depth=True)

num_epochs = 20
model.to(device)

In [ ]:
train_loss_list, validation_loss_list = [], []
pretrained = False
model_save_path = '../models/checkpoints/Image2PlantArchitecture_20240922_Normalize_False.pth'
if pretrained:
    model.load_state_dict(torch.load(model_save_path))

In [ ]:
from plant_tokenizer import params_SOS_token_padded, SOS_token, EOS_token
from plant_tokenizer import token2vec_new as token2vec
from plant_tokenizer import vec2token_new as vec2token
from utils import get_tgt_mask



def predict(model, images, max_length=15, SOS_token=2, EOS_token=3, device="cuda"):
    # Assuming 'device' is defined elsewhere in your code and is a CUDA device.
    y_input = torch.tensor(params_SOS_token_padded, dtype=torch.float32)
    
    # Change to 1,1,15
    y_input = y_input.unsqueeze(0).unsqueeze(0)
    y_input = y_input.to(device)
    for i in range(max_length):
        # Get source mask
        tgt_mask = get_tgt_mask(y_input.size(1)).to(device)
        
        try:
            with torch.no_grad():
                pred = model(images, y_input, tgt_mask)
        except Exception as e:
            print(e)
            print(f"Error in {i} iteration")
            break
        label_p = pred[:,:,:seq_dim]
        label = label_p.topk(1)[1].view(-1)[-1].item()  # num with highest probability
        params = pred[:,:,seq_dim:]

        # Stop if model predicts end of sentencplant_structure_vit_transformer_withpsudodepth_paramEste
        if label == EOS_token or label == PAD_token:
            break

        # Make next tensor using label and params
        next_item = torch.cat((torch.tensor([[label]], dtype=torch.float32, device=device), params[-1]), dim=1).unsqueeze(0)

        # Concatenate previous input with predicted best word
        y_input = torch.cat((y_input, next_item), dim=1)

    return y_input.squeeze(0).tolist()

model.eval()

#test_dataset = PlantDataset("../data/generated_renamed", plot=["004"],stages=["023"],
                            # transform=transform, use_depth=True, preload=False, process_leaf=False)
test_dataset = PlantDataset("../data/generated_dataset_Sep22_Black", plot=["004"],stages=["003"],
                            transform=transform, use_depth=True, preload=False, process_leaf=False)
gen_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

#for idx, (image, out, lengths) in enumerate(gen_dataloader):
for idx, (image, out, lengths) in enumerate(test_dataset):
    
    if image.dim() == 3:
        image = image.unsqueeze(0)

    print(test_dataset.image_paths[idx])
    print("Ground truth")
    print(test_dataset.plant_string_raw)

    # Print out file name

    image_paths = image.to(device)

    # Plot the image
    # Draw the image
    image_vis = image.squeeze(0).permute(1,2,0).cpu().numpy()
    img = cv2.normalize(np.array(image_vis), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    #img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    image_vis = img.astype(np.uint8)
    # plt.imshow(image_vis)
    result = predict(model, image_paths, max_length=2048, SOS_token=SOS_token, EOS_token=EOS_token)
    # print(f"Example {idx}")
    # print(f"Input Structure: {token2vec(out.squeeze(0).tolist())}")
    # print(f"Input Token: {out.squeeze(0).tolist()}")
    # print(f"Continuation Token: {result}")
    # print(f"Continuation Structure: {token2vec(result)}")
    # print()
    break

out = torch.tensor(out).to(device)
ground_truth = out.squeeze(0).tolist()
if 1:
    # Token to structure
    plant_vec = token2vec(result, normalize=False)
    # print(plant_vec)
    # Plant vec to string
    plant_string = vec2string([plant_vec])
else:
    # Token to structure
    plant_vec = token2vec(ground_truth, normalize=False)
    # print(plant_vec)
    # Plant vec to string
    plant_string = vec2string([plant_vec])
print("Output")
print(plant_string)

# plant_vec to xml
#xml_string = vec2xml(plant_vec)
#print(pretty_print_xml(xml_string))


# print(plant_string)
# Save plant string to text
plant_string_file_name = "plant_string.txt"
with open(plant_string_file_name, "w") as f:
    f.write(plant_string)



# Generate plant image
from plantstring2model import plantstring2model
from image_process import process_leaf_image
import cv2
import matplotlib.pyplot as plt
p2m = plantstring2model(program_path="PlantString2Model/build",program_name="PlantString2Model",display=":11.0", height=1.0)

# Run 
p2m.run(plantstring_path=os.path.abspath("plant_string.txt"))
generated_image_path = "output/plant_string_top.jpeg"

# Load the image
img = cv2.imread(generated_image_path)
leaf_area, plant_width, plant_height, leaf_img, _ = process_leaf_image(img,sqaure_crop=True, thr = 0.2)
# Plot the processed image
# plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))


# Plot the original and processed images
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Original image
axes[0].imshow(image_vis[:,:,0:3])
axes[0].set_title("Input Image")
axes[0].axis('off')

# Processed image
img = cv2.resize(img,(224,224))
axes[1].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Output model")
axes[1].axis('off')

